Guardrails help you build safe, complaint AI application by validating and filtering content at key point in your agent's execution.

They are implemented as middleware that intercepts execution:

. Beforethe agent starts(input gurardrails)
. After it completes (output guardrails)
. Around model and tool calls

### Common Use Cases :

Use case                   Examples
PII Leakage prevention      Redact emails/credit cards before logging
Prompt injection blocking   Detect adversarial inputs
Harmful content filtering   Block dangerous requests
Business rule enforcement   Require approval for financial ops
Output quality validation   Ensure response meets safety standard

### Deterministic Guardrails

. Rule-based : regex, keyword marching, explicit checks
. Fast, predictable, cost-effective
. May miss nuanced violations

### Model-Based Guardrails
. Uses LLM / classifiers for semantic understanding
. Catches subtle/nuanced isses
. slower and more expensive

In [ ]:
import re

#-----------Deterministic apprach ---------
def deterministic_guardrail(text: str) -> bool:
    """ Returns True if content is blocked."""
    banned_keywords = ["hack", "expliot", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of france?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ==========")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = " Blocked " if blocked else "ALLOWED"
    print(f"{status}: {inp}")


=== Deterministic Guard Demo ==========
 Blocked : How do I hack into a database?
ALLOWED: What is the capital of france?
 Blocked : Explain how malware spreads


In [11]:
from langchain_openai import ChatOpenAI

# -- Model-based appraoch ---

def model_based_guardrail(text: str) -> str:
    """ Users an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    prompt = f"""Evaluate the following input safe to process. Respond with only SAFE or UNSAFE.

    Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])    
    return result.content.strip()

print("=== Model-based Guardrail Demo ==========")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

=== Model-based Guardrail Demo ==========
UNSAFE: How do I hack into a database?
SAFE: What is the capital of france?
SAFE: Explain how malware spreads


### Section 3 : Built in Guardrail - PII Detection Middleware

Langchain provide buil-in PIIMiddleware for detecting and handling Personalily Identifiable Information (PII)

Support PII Types:


Type         Example
email        user@example.com
credit_card  5105-1051-510-5100
ip           192.168.1.1
mac_address  00:1A:2B:3C:4D:5E
url          https://secret-site.com

Strategies:

Strategy     Result
redact       [REDACTED_EMAIL]
mask         *****_****_****-1234
hash         a8f5167......
block Raises an exception

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information ."""
    return f"Customer info for query: {query}"

#Create agent with PII Middleware
agent = create_agent(
    model="gpt-4o-mini",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware()]
)

In [13]:
from pathlib import Path
from openai import OpenAI
from IPython.display import Audio, display

client = OpenAI()

business_name = "Elysian Earth"
contact_number = "9958714800"

message = f"""
Hello! Welcome to Elysian Earth.
Looking for handloom fashion for everyday wear or special occasions?
Message us to explore our collection, ask about prices,
and find something that suits your style.
For our latest catalogue, contact 9958714800
and send the word CATALOGUE.
We look forward to helping you!
"""

output_file = Path(r"c:\ML\3 - RAG\handloom_voice_note.mp3")

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input=message,
    instructions="Speak warmly and clearly, like a friendly boutique owner.",
) as response:
    response.stream_to_file(output_file)

display(Audio(filename=str(output_file)))
print(f"Voice note saved: {output_file}")

Voice note saved: c:\ML\3 - RAG\handloom_voice_note.mp3
